# 01 Events base and gold tables

Notebook 01 builds the reusable BigQuery and Drive gold table layer for the GA4 capstone project. The raw GA4 export contains nested event parameters, ecommerce fields, and repeated item arrays, so this notebook converts the event stream into clean analytical tables that downstream notebooks can reuse.

A key design choice is that the modelling feature tables are restricted to activity observed before the first purchase event in each session. The purchase label and order records are still retained, but they are kept separate for evaluation, reporting, and audit rather than being merged into the modelling features.

---

## Purpose

- Create the reusable `events_base` table from the public GA4 ecommerce export
- Build session, funnel-step, purchase-label, item, order, and weekly fact tables at clear grains
- Standardise clean channel fields early, including `channel_segment`
- Build pre-purchase behavioural, funnel, and item features for Notebook 03
- Export the processed parquet files required by the modelling notebook
- Export the first clean Power BI fact tables produced by this notebook
- Save data quality checks and a table dictionary for reproducibility

---

## Process

1. Set up the Colab, Drive, and BigQuery environment so the notebook can run from a clean runtime.
2. Define the shared folders, date window, helper functions, and destination table names.
3. Create the destination BigQuery dataset and define all canonical table names in one place.
4. Build the event-level base table by flattening useful GA4 fields while preserving the event grain.
5. Build the gold tables at session, funnel-step, label, item, order, and weekly fact grains.
6. Restrict modelling features to pre-purchase activity for purchase sessions, while keeping purchase events for labels and order audit.
7. Export the required downstream parquet and Power BI CSV outputs, then save data quality checks.

# 1.0 Data imports and environment setup

In [1]:
#------------------------------------------------------------------------------
# 1.1 Install packages
#------------------------------------------------------------------------------
%pip -q install db-dtypes pyarrow pandas-gbq google-cloud-bigquery

#------------------------------------------------------------------------------
# 1.2 Import libraries and mount Drive
#------------------------------------------------------------------------------
from pathlib import Path
import json
import numpy as np, pandas as pd, pandas_gbq

from google.colab import drive, auth
from google.cloud import bigquery
from IPython.display import display

drive.mount("/content/drive", force_remount=True)
auth.authenticate_user()

import google.auth
CREDS, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])

# Set notebook display options
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Set the BigQuery project and client
BQ_PROJECT_ID = "ga4-project-490603"
pandas_gbq.context.project = BQ_PROJECT_ID
pandas_gbq.context.location = "US"
bq_client = bigquery.Client(project=BQ_PROJECT_ID, credentials=CREDS, location="US")


Mounted at /content/drive


# 2.0 Core config and helper functions

Core configuration is kept near the top so the rest of the notebook uses one consistent set of paths, dates, and helper functions. The selected GA4 date window is recorded here so the extraction period is visible before any BigQuery tables are created.

**Process**:
1. Define the shared Drive folders used for SQL scripts, data quality outputs, Power BI extracts, and processed parquet files.
2. Set the final `START_DATE` and `END_DATE` values used by all BigQuery wildcard table queries.
3. Define small helper functions for running BigQuery SQL, saving SQL files, exporting query results, and overwriting JSON outputs.
4. Save run metadata record for reproducibility.

In [2]:
#------------------------------------------------------------------------------
# 2.1 Core config and Drive folders
#------------------------------------------------------------------------------
# Define BigQuery and Drive paths
GA4_PUBLIC_DATASET_TABLE = "bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*"
BQ_DATASET_ID = "ga4_capstone"

CAPSTONE_DIR = Path("/content/drive/MyDrive/Capstone_Project")
OUTPUT_DIR = CAPSTONE_DIR / "outputs"
SQL_DIR = OUTPUT_DIR / "sql"
DQ_DIR = OUTPUT_DIR / "data_quality"
PBI_LATEST_DIR = CAPSTONE_DIR / "powerbi_extracts" / "latest"
PROCESSED_DIR = CAPSTONE_DIR / "data" / "processed"
RUN_METADATA_PATH = OUTPUT_DIR / "run_metadata.json"

# Create required output folders
for p in [CAPSTONE_DIR, OUTPUT_DIR, SQL_DIR, DQ_DIR, PBI_LATEST_DIR, PROCESSED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("CAPSTONE_DIR     :", CAPSTONE_DIR)
print("PBI_LATEST_DIR   :", PBI_LATEST_DIR)
print("PROCESSED_DIR    :", PROCESSED_DIR)
print("RUN_METADATA_PATH:", RUN_METADATA_PATH)


CAPSTONE_DIR     : /content/drive/MyDrive/Capstone_Project
PBI_LATEST_DIR   : /content/drive/MyDrive/Capstone_Project/powerbi_extracts/latest
PROCESSED_DIR    : /content/drive/MyDrive/Capstone_Project/data/processed
RUN_METADATA_PATH: /content/drive/MyDrive/Capstone_Project/outputs/run_metadata.json


In [3]:
#------------------------------------------------------------------------------
# 2.2 Set date window and define helpers
#------------------------------------------------------------------------------
# Set the final GA4 date windows
START_DATE = "20201101"
END_DATE = "20210228"

def print_banner(title):
    print("\n" + "-" * 78); print(title); print("-" * 78)

# Run a BigQuery SELECT query and return a dataframe
def run_bq(sql):
    return pandas_gbq.read_gbq(sql, project_id=BQ_PROJECT_ID, location="US", progress_bar_type=None)

# Save JSON outputs with safe conversion for BigQuery and pandas types
def save_json_overwrite(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

# Save SQL to Drive before running it in BigQuery
def save_sql_and_run(file_name, sql_text):
    sql_path = SQL_DIR / file_name
    sql_path.write_text(sql_text, encoding="utf-8")
    bq_client.query(sql_text).result()
    print("Saved SQL ->", sql_path)

# Export a query result to CSV and/or parquet
def export_query(sql, csv_path=None, parquet_path=None, date_cols=None, datetime_cols=None, string_cols=None):
    df = run_bq(sql)
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(num_cols) > 0:
        df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)
    # Format selected date columns before saving
    for col in date_cols or []:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce").dt.strftime("%Y-%m-%d")
    # Convert selected datetime columns before saving
    for col in datetime_cols or []:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    # Convert selected ID fields to strings before saving
    for col in string_cols or []:
        if col in df.columns:
            df[col] = df[col].astype("string")
    # Save CSV output when requested
    if csv_path is not None:
        csv_path.parent.mkdir(parents=True, exist_ok=True); df.to_csv(csv_path, index=False)
    # Save parquet output when requested
    if parquet_path is not None:
        parquet_path.parent.mkdir(parents=True, exist_ok=True); df.to_parquet(parquet_path, index=False)
    return df

# Create the target BigQuery dataset if needed
def ensure_dataset(project_id, dataset_id, location="US"):
    dataset_ref = bigquery.Dataset(f"{project_id}.{dataset_id}")
    dataset_ref.location = location
    bq_client.create_dataset(dataset_ref, exists_ok=True)

print("START_DATE:", START_DATE)
print("END_DATE  :", END_DATE)

# Save the current run settings for later notebooks
run_metadata_current = {
    "notebook": "01_events_base_and_gold_tables",
    "START_DATE": START_DATE,
    "END_DATE": END_DATE,
    "bq_project_id": BQ_PROJECT_ID,
    "bq_dataset_id": BQ_DATASET_ID,
    "source_table": GA4_PUBLIC_DATASET_TABLE,
    "capstone_dir": str(CAPSTONE_DIR)}
save_json_overwrite(RUN_METADATA_PATH, run_metadata_current)
print("Updated run metadata ->", RUN_METADATA_PATH)

START_DATE: 20201101
END_DATE  : 20210228
Updated run metadata -> /content/drive/MyDrive/Capstone_Project/outputs/run_metadata.json


# 3.0 Create dataset and table names

The BigQuery destination dataset and table constants are defined before any table building SQL is executed. Keeping these names in one place reduces the chance of writing to inconsistent table names and makes the downstream dependencies easier to check.

**Process**:
1. Create the `ga4_capstone` BigQuery dataset if it does not already exist.
2. Define canonical names for the event base table, gold tables, orders table, and weekly fact tables.
3. Reuse these constants throughout the SQL blocks so each table is rebuilt in a predictable location.

In [4]:
#------------------------------------------------------------------------------
# 3.1 Create the dataset and table name constants
#------------------------------------------------------------------------------
print_banner("3.1 Create dataset and table name constants")

ensure_dataset(BQ_PROJECT_ID, BQ_DATASET_ID, location="US")

# Store BigQuery table names
EVENTS_BASE_TABLE = f"{BQ_PROJECT_ID}.{BQ_DATASET_ID}.events_base"
SESSIONS_TABLE = f"{BQ_PROJECT_ID}.{BQ_DATASET_ID}.sessions"
FUNNEL_STEPS_TABLE = f"{BQ_PROJECT_ID}.{BQ_DATASET_ID}.funnel_steps"
SESSION_LABEL_TABLE = f"{BQ_PROJECT_ID}.{BQ_DATASET_ID}.session_label"
FUNNEL_FEATURES_TABLE = f"{BQ_PROJECT_ID}.{BQ_DATASET_ID}.funnel_features"
ITEMS_SESSION_TABLE = f"{BQ_PROJECT_ID}.{BQ_DATASET_ID}.items_session"
ORDERS_TABLE = f"{BQ_PROJECT_ID}.{BQ_DATASET_ID}.orders"
FACT_SESSIONS_WEEKLY_TABLE = f"{BQ_PROJECT_ID}.{BQ_DATASET_ID}.fact_sessions_weekly"
FACT_FUNNEL_WEEKLY_TABLE = f"{BQ_PROJECT_ID}.{BQ_DATASET_ID}.fact_funnel_weekly"

print("Dataset ready ->", f"{BQ_PROJECT_ID}.{BQ_DATASET_ID}")



------------------------------------------------------------------------------
3.1 Create dataset and table name constants
------------------------------------------------------------------------------
Dataset ready -> ga4-project-490603.ga4_capstone


# 4.0 Create the event-level base table

The event-level base table is the reusable staging layer for the rest of the project. GA4 stores important fields inside nested `event_params`, `ecommerce`, and `items` structures, so this table extracts the most useful repeated fields once while keeping one row per event.

**Process**:
1. Read the selected GA4 wildcard date window from the public ecommerce export.
2. Extract session identifiers, event timestamps, engagement fields, page context, device fields, geography fields, and acquisition fields.
3. Retain the nested `items` array so later item features can be rebuilt safely from pre-purchase item rows.
4. Save event-base quality JSON that records row counts, user counts, session coverage, item coverage, and purchase event volume.
5. Display a small sample for inspection without saving the sample as a final output.

In [5]:
#------------------------------------------------------------------------------
# 4.1 Create the event-level base table
#------------------------------------------------------------------------------
print_banner("4.1 Create event-level base table")

# Build one event level staging table from the selected GA4 window
events_base_sql = f"""
CREATE OR REPLACE TABLE `{EVENTS_BASE_TABLE}` AS
SELECT
  event_date,
  event_timestamp,
  TIMESTAMP_MICROS(event_timestamp) AS event_time,
  event_name,
  SAFE_CAST(user_pseudo_id AS STRING) AS user_pseudo_id,
  user_id,

  -- Sessionisation fields
  (SELECT ep.value.int_value FROM UNNEST(event_params) ep WHERE ep.key = 'ga_session_id') AS ga_session_id,
  (SELECT ep.value.int_value FROM UNNEST(event_params) ep WHERE ep.key = 'ga_session_number') AS ga_session_number,
  COALESCE(
    (SELECT ep.value.int_value FROM UNNEST(event_params) ep WHERE ep.key = 'session_engaged'),
    SAFE_CAST((SELECT ep.value.string_value FROM UNNEST(event_params) ep WHERE ep.key = 'session_engaged') AS INT64)
  ) AS session_engaged_flag,
  (SELECT ep.value.int_value FROM UNNEST(event_params) ep WHERE ep.key = 'engagement_time_msec') AS engagement_time_msec,

  -- Page context
  (SELECT ep.value.string_value FROM UNNEST(event_params) ep WHERE ep.key = 'page_location') AS page_location,
  (SELECT ep.value.string_value FROM UNNEST(event_params) ep WHERE ep.key = 'page_referrer') AS page_referrer,
  (SELECT ep.value.string_value FROM UNNEST(event_params) ep WHERE ep.key = 'page_title') AS page_title,
  event_dimensions.hostname AS hostname,

  -- Event-level acquisition fields
  (SELECT ep.value.string_value FROM UNNEST(event_params) ep WHERE ep.key = 'source') AS ep_source,
  (SELECT ep.value.string_value FROM UNNEST(event_params) ep WHERE ep.key = 'medium') AS ep_medium,
  (SELECT ep.value.string_value FROM UNNEST(event_params) ep WHERE ep.key = 'campaign') AS ep_campaign,
  (SELECT ep.value.string_value FROM UNNEST(event_params) ep WHERE ep.key = 'term') AS ep_term,
  (SELECT ep.value.string_value FROM UNNEST(event_params) ep WHERE ep.key = 'content') AS ep_content,

  -- Device and geo fields
  device.category AS device_category,
  device.operating_system AS device_os,
  device.web_info.browser AS device_browser,
  geo.country AS geo_country,
  geo.region AS geo_region,
  geo.city AS geo_city,

  -- First acquisition traffic-source fields
  traffic_source.source AS traffic_source_source,
  traffic_source.medium AS traffic_source_medium,
  traffic_source.name AS traffic_source_name,

  -- Ecommerce and item fields
  ecommerce,
  items,
  platform,
  stream_id
FROM `{GA4_PUBLIC_DATASET_TABLE}`
WHERE _TABLE_SUFFIX BETWEEN '{START_DATE}' AND '{END_DATE}';
"""

save_sql_and_run("events_base_create_table.sql", events_base_sql)
print("Created table ->", EVENTS_BASE_TABLE)



------------------------------------------------------------------------------
4.1 Create event-level base table
------------------------------------------------------------------------------
Saved SQL -> /content/drive/MyDrive/Capstone_Project/outputs/sql/events_base_create_table.sql
Created table -> ga4-project-490603.ga4_capstone.events_base


In [6]:
#------------------------------------------------------------------------------
# 4.2 Display a small quality summary and sample
#------------------------------------------------------------------------------
print_banner("4.2 Display events_base quality summary and sample")

# Summarise the base table coverage
events_base_quality_sql = f"""
SELECT
  COUNT(*) AS n_rows,
  COUNT(DISTINCT user_pseudo_id) AS n_users,
  COUNT(DISTINCT CONCAT(user_pseudo_id, '-', CAST(ga_session_id AS STRING))) AS approx_user_session_pairs,
  COUNTIF(ga_session_id IS NOT NULL) AS n_rows_with_session_id,
  COUNTIF(ARRAY_LENGTH(items) > 0) AS n_rows_with_items,
  COUNTIF(event_name = 'purchase') AS n_purchase_rows,
  MIN(event_date) AS min_event_date,
  MAX(event_date) AS max_event_date
FROM `{EVENTS_BASE_TABLE}`;
"""

# Pull a small ordered sample for inspection
events_base_sample_sql = f"""
SELECT *
FROM `{EVENTS_BASE_TABLE}`
ORDER BY event_timestamp
LIMIT 20;
"""

# Run queries
events_base_quality_df = run_bq(events_base_quality_sql)
events_base_sample_df = run_bq(events_base_sample_sql)

save_json_overwrite(DQ_DIR / "events_base_quality.json", events_base_quality_df.iloc[0].to_dict())

display(events_base_quality_df)
display(events_base_sample_df.head(10))



------------------------------------------------------------------------------
4.2 Display events_base quality summary and sample
------------------------------------------------------------------------------


,n_rows,n_users,approx_user_session_pairs,n_rows_with_session_id,n_rows_with_items,n_purchase_rows,min_event_date,max_event_date
0,4295584,270154,360129,4295584,512346,5692,20201101,20210131


,event_date,event_timestamp,event_time,event_name,user_pseudo_id,user_id,ga_session_id,ga_session_number,session_engaged_flag,engagement_time_msec,page_location,page_referrer,page_title,hostname,ep_source,ep_medium,ep_campaign,ep_term,ep_content,device_category,device_os,device_browser,geo_country,geo_region,geo_city,traffic_source_source,traffic_source_medium,traffic_source_name,ecommerce,items,platform,stream_id
0,20201101,1604188804579566,2020-11-01 00:00:04.579566+00:00,first_visit,82023966.6010891291,None,1090192382,1,1,<NA>,https://shop.googlemerchandisestore.com/,None,Home,None,None,None,None,None,None,desktop,Macintosh,Chrome,Canada,Ontario,Toronto,(direct),(none),(direct),"{'total_item_quantity': None, 'purchase_revenue_in_usd': None, 'purchase_revenue': None, 'refund_value_in_usd': None, 'refund_value': None, 'shipping_value_...",[],WEB,2100450278
1,20201101,1604188804579566,2020-11-01 00:00:04.579566+00:00,session_start,82023966.6010891291,None,1090192382,1,<NA>,<NA>,https://shop.googlemerchandisestore.com/,None,Home,None,None,None,None,None,None,desktop,Macintosh,Chrome,Canada,Ontario,Toronto,(direct),(none),(direct),"{'total_item_quantity': None, 'purchase_revenue_in_usd': None, 'purchase_revenue': None, 'refund_value_in_usd': None, 'refund_value': None, 'shipping_value_...",[],WEB,2100450278
2,20201101,1604188804579566,2020-11-01 00:00:04.579566+00:00,page_view,82023966.6010891291,None,1090192382,1,0,<NA>,https://shop.googlemerchandisestore.com/,None,Home,None,None,None,None,None,None,desktop,Macintosh,Chrome,Canada,Ontario,Toronto,(direct),(none),(direct),"{'total_item_quantity': None, 'purchase_revenue_in_usd': None, 'purchase_revenue': None, 'refund_value_in_usd': None, 'refund_value': None, 'shipping_value_...",[],WEB,2100450278
3,20201101,1604188806427147,2020-11-01 00:00:06.427147+00:00,first_visit,7754124.2052719725,None,3291659989,1,1,<NA>,https://shop.googlemerchandisestore.com/Google+Redesign/Lifestyle/Bags,None,Bags | Lifestyle | Google Merchandise Store,None,None,None,None,None,None,mobile,Web,<Other>,United States,Illinois,(not set),google,cpc,<Other>,"{'total_item_quantity': None, 'purchase_revenue_in_usd': None, 'purchase_revenue': None, 'refund_value_in_usd': None, 'refund_value': None, 'shipping_value_...",[],WEB,2100450278
4,20201101,1604188806427147,2020-11-01 00:00:06.427147+00:00,session_start,7754124.2052719725,None,3291659989,1,<NA>,<NA>,https://shop.googlemerchandisestore.com/Google+Redesign/Lifestyle/Bags,None,Bags | Lifestyle | Google Merchandise Store,None,None,None,None,None,None,mobile,Web,<Other>,United States,Illinois,(not set),google,cpc,<Other>,"{'total_item_quantity': None, 'purchase_revenue_in_usd': None, 'purchase_revenue': None, 'refund_value_in_usd': None, 'refund_value': None, 'shipping_value_...",[],WEB,2100450278
5,20201101,1604188806427147,2020-11-01 00:00:06.427147+00:00,page_view,7754124.2052719725,None,3291659989,1,0,<NA>,https://shop.googlemerchandisestore.com/Google+Redesign/Lifestyle/Bags,None,Bags | Lifestyle | Google Merchandise Store,None,google,cpc,<Other>,None,None,mobile,Web,<Other>,United States,Illinois,(not set),google,cpc,<Other>,"{'total_item_quantity': None, 'purchase_revenue_in_usd': None, 'purchase_revenue': None, 'refund_value_in_usd': None, 'refund_value': None, 'shipping_value_...",[],WEB,2100450278
6,20201101,1604188809674418,2020-11-01 00:00:09.674418+00:00,view_promotion,82023966.6010891291,None,1090192382,1,0,88,https://shop.googlemerchandisestore.com/,None,Home,None,None,None,None,None,None,desktop,Macintosh,Chrome,Canada,Ontario,Toronto,(direct),(none),(direct),"{'total_item_quantity': None, 'purchase_revenue_in_usd': None, 'purchase_revenue': None, 'refund_value_in_usd': None, 'refund_value': None, 'shipping_value_...",[],WEB,2100450278
7,20201101,1604188814982984,2020-11-01 00:00:14.982984+00:00,scroll,82023966.6010891291,None,1090192382,1,0,5311,https://shop.googlemerchandisestore.com/,None,Home,None,None,None,None,None,None,desktop,Macinto

# 5.0 Create gold tables

The gold tables turn the event-level base table into reusable analytical tables at clear grains. The main modelling design choice is that behavioural, funnel, and item features are calculated only from events before the first purchase event in a session, while the purchase label and orders table are kept separately.

**Process**:
1. Build the `sessions` table at one row per user session, using only pre-purchase events for behavioural counts, engagement time, and duration in purchase sessions.
2. Standardise `channel_source`, `channel_medium`, `channel_campaign`, and `channel_segment` in the session table so downstream notebooks do not rebuild channel logic.
3. Build the `funnel_steps` table with first reach timestamps for funnel milestones, keeping the first purchase event for labelling but excluding repeated post-purchase funnel activity.
4. Build the `session_label` table as a separate binary purchase label so target information does not become part of the modelling features.
5. Build `funnel_features` and `items_session` using pre-purchase activity only. This keeps valid pre-purchase funnel progress while avoiding purchase-event and post-purchase leakage.
6. Build the `orders` table
7. Run a validation check to confirm session counts, item counts, and funnel steps align with the pre-purchase feature design.

In [7]:
#------------------------------------------------------------------------------
# 5.1 Create sessions, funnel_steps, and session_label
#------------------------------------------------------------------------------
print_banner("5.1 Create sessions, funnel_steps, and session_label")

# Create the session table with pre-purchase behavioural counts
sessions_sql = f"""
CREATE OR REPLACE TABLE `{SESSIONS_TABLE}` AS
WITH base AS (
  SELECT
    user_pseudo_id, ga_session_id, ga_session_number, event_timestamp, event_name,
    engagement_time_msec, session_engaged_flag,
    device_category, device_os, device_browser, geo_country, geo_region, geo_city,
    ep_source, ep_medium, ep_campaign, ep_term,
    traffic_source_source, traffic_source_medium, traffic_source_name,
    CONCAT(user_pseudo_id, '-', CAST(ga_session_id AS STRING)) AS user_session_key
  FROM `{EVENTS_BASE_TABLE}`
  WHERE ga_session_id IS NOT NULL
),
first_purchase AS (
  SELECT user_session_key, MIN(event_timestamp) AS first_purchase_ts
  FROM base
  WHERE event_name = 'purchase'
  GROUP BY user_session_key
),
labelled AS (
  SELECT
    b.*,
    p.first_purchase_ts,
    p.first_purchase_ts IS NULL OR b.event_timestamp < p.first_purchase_ts AS is_model_event
  FROM base b
  LEFT JOIN first_purchase p
    ON b.user_session_key = p.user_session_key
),
session_rollup AS (
  SELECT
    user_pseudo_id,
    ga_session_id,
    user_session_key,
    ARRAY_AGG(ga_session_number IGNORE NULLS ORDER BY event_timestamp ASC LIMIT 1)[SAFE_OFFSET(0)] AS ga_session_number,

    MIN(event_timestamp) AS full_session_start_ts,
    MIN(IF(is_model_event, event_timestamp, NULL)) AS feature_start_ts,
    MAX(IF(is_model_event, event_timestamp, NULL)) AS feature_end_ts,

    COUNTIF(is_model_event) AS n_events,
    COUNT(DISTINCT IF(is_model_event, event_name, NULL)) AS n_distinct_event_names,
    SUM(IF(is_model_event, COALESCE(engagement_time_msec, 0), 0)) AS engagement_time_msec_sum,
    MAX(IF(is_model_event, COALESCE(session_engaged_flag, 0), 0)) AS session_engaged_flag_max,

    ARRAY_AGG(device_category IGNORE NULLS ORDER BY event_timestamp ASC LIMIT 1)[SAFE_OFFSET(0)] AS device_category,
    ARRAY_AGG(device_os IGNORE NULLS ORDER BY event_timestamp ASC LIMIT 1)[SAFE_OFFSET(0)] AS device_os,
    ARRAY_AGG(device_browser IGNORE NULLS ORDER BY event_timestamp ASC LIMIT 1)[SAFE_OFFSET(0)] AS device_browser,
    ARRAY_AGG(geo_country IGNORE NULLS ORDER BY event_timestamp ASC LIMIT 1)[SAFE_OFFSET(0)] AS geo_country,
    ARRAY_AGG(geo_region IGNORE NULLS ORDER BY event_timestamp ASC LIMIT 1)[SAFE_OFFSET(0)] AS geo_region,
    ARRAY_AGG(geo_city IGNORE NULLS ORDER BY event_timestamp ASC LIMIT 1)[SAFE_OFFSET(0)] AS geo_city,

    COALESCE(
      ARRAY_AGG(NULLIF(TRIM(ep_source), '') IGNORE NULLS ORDER BY event_timestamp ASC LIMIT 1)[SAFE_OFFSET(0)],
      ARRAY_AGG(NULLIF(TRIM(traffic_source_source), '') IGNORE NULLS ORDER BY event_timestamp ASC LIMIT 1)[SAFE_OFFSET(0)]
    ) AS channel_source_raw,

    COALESCE(
      ARRAY_AGG(NULLIF(TRIM(ep_medium), '') IGNORE NULLS ORDER BY event_timestamp ASC LIMIT 1)[SAFE_OFFSET(0)],
      ARRAY_AGG(NULLIF(TRIM(traffic_source_medium), '') IGNORE NULLS ORDER BY event_timestamp ASC LIMIT 1)[SAFE_OFFSET(0)]
    ) AS channel_medium_raw,

    COALESCE(
      ARRAY_AGG(NULLIF(TRIM(ep_campaign), '') IGNORE NULLS ORDER BY event_timestamp ASC LIMIT 1)[SAFE_OFFSET(0)],
      ARRAY_AGG(NULLIF(TRIM(traffic_source_name), '') IGNORE NULLS ORDER BY event_timestamp ASC LIMIT 1)[SAFE_OFFSET(0)]
    ) AS channel_campaign_raw,

    ARRAY_AGG(NULLIF(TRIM(ep_term), '') IGNORE NULLS ORDER BY event_timestamp ASC LIMIT 1)[SAFE_OFFSET(0)] AS channel_term
  FROM labelled
  GROUP BY user_pseudo_id, ga_session_id, user_session_key
)
SELECT
  user_pseudo_id,
  ga_session_id,
  user_session_key,
  ga_session_number,
  COALESCE(feature_start_ts, full_session_start_ts) AS session_start_ts,
  COALESCE(feature_end_ts, feature_start_ts, full_session_start_ts) AS session_end_ts,
  TIMESTAMP_MICROS(COALESCE(feature_start_ts, full_session_start_ts)) AS session_start_time,
  TIMESTAMP_MICROS(COALESCE(feature_end_ts, feature_start_ts, full_session_start_ts)) AS session_end_time,
  DATE(TIMESTAMP_MICROS(full_session_start_ts)) AS session_date,
  DATE_TRUNC(DATE(TIMESTAMP_MICROS(full_session_start_ts)), WEEK(MONDAY)) AS week_start_date,
  CASE
    WHEN feature_start_ts IS NULL OR feature_end_ts IS NULL THEN 0
    ELSE SAFE_DIVIDE(feature_end_ts - feature_start_ts, 1000000.0)
  END AS session_duration_seconds,
  n_events,
  n_distinct_event_names,
  engagement_time_msec_sum,
  session_engaged_flag_max,
  COALESCE(device_category, '(not set)') AS device_category,
  device_os,
  device_browser,
  COALESCE(geo_country, '(not set)') AS geo_country,
  geo_region,
  geo_city,
  COALESCE(channel_source_raw, '(not set)') AS channel_source,
  COALESCE(channel_medium_raw, '(not set)') AS channel_medium,
  COALESCE(channel_campaign_raw, '(not set)') AS channel_campaign,
  CONCAT(COALESCE(channel_source_raw, '(not set)'), ' / ', COALESCE(channel_medium_raw, '(not set)')) AS channel_segment,
  channel_term
FROM session_rollup;
"""

save_sql_and_run("sessions_create_table.sql", sessions_sql)
print("Created table ->", SESSIONS_TABLE)

# Create funnel step timestamps and keep the first purchase event
funnel_steps_sql = f"""
CREATE OR REPLACE TABLE `{FUNNEL_STEPS_TABLE}` AS
WITH base AS (
  SELECT
    user_pseudo_id,
    ga_session_id,
    CONCAT(user_pseudo_id, '-', CAST(ga_session_id AS STRING)) AS user_session_key,
    event_name,
    event_timestamp
  FROM `{EVENTS_BASE_TABLE}`
  WHERE ga_session_id IS NOT NULL
    AND event_name IN ('view_item','add_to_cart','begin_checkout','add_shipping_info','add_payment_info','purchase')
),
first_purchase AS (
  SELECT user_session_key, MIN(event_timestamp) AS first_purchase_ts
  FROM base
  WHERE event_name = 'purchase'
  GROUP BY user_session_key
),
safe_steps AS (
  SELECT b.*
  FROM base b
  LEFT JOIN first_purchase p
    ON b.user_session_key = p.user_session_key
  WHERE p.first_purchase_ts IS NULL
     OR b.event_timestamp < p.first_purchase_ts
     OR (b.event_name = 'purchase' AND b.event_timestamp = p.first_purchase_ts)
)
SELECT
  user_pseudo_id,
  ga_session_id,
  event_name AS funnel_step,
  MIN(event_timestamp) AS first_step_ts,
  TIMESTAMP_MICROS(MIN(event_timestamp)) AS first_step_time,
  DATE(TIMESTAMP_MICROS(MIN(event_timestamp))) AS first_step_date
FROM safe_steps
GROUP BY user_pseudo_id, ga_session_id, funnel_step;
"""

save_sql_and_run("funnel_steps_create_table.sql", funnel_steps_sql)
print("Created table ->", FUNNEL_STEPS_TABLE)

# Keep the purchase label separate from modelling features
session_label_sql = f"""
CREATE OR REPLACE TABLE `{SESSION_LABEL_TABLE}` AS
SELECT
  s.user_pseudo_id,
  s.ga_session_id,
  IF(COUNTIF(f.funnel_step = 'purchase') > 0, 1, 0) AS y_purchase
FROM `{SESSIONS_TABLE}` s
LEFT JOIN `{FUNNEL_STEPS_TABLE}` f
  ON s.user_pseudo_id = f.user_pseudo_id AND s.ga_session_id = f.ga_session_id
GROUP BY s.user_pseudo_id, s.ga_session_id;
"""

save_sql_and_run("session_label_create_table.sql", session_label_sql)
print("Created table ->", SESSION_LABEL_TABLE)



------------------------------------------------------------------------------
5.1 Create sessions, funnel_steps, and session_label
------------------------------------------------------------------------------
Saved SQL -> /content/drive/MyDrive/Capstone_Project/outputs/sql/sessions_create_table.sql
Created table -> ga4-project-490603.ga4_capstone.sessions
Saved SQL -> /content/drive/MyDrive/Capstone_Project/outputs/sql/funnel_steps_create_table.sql
Created table -> ga4-project-490603.ga4_capstone.funnel_steps
Saved SQL -> /content/drive/MyDrive/Capstone_Project/outputs/sql/session_label_create_table.sql
Created table -> ga4-project-490603.ga4_capstone.session_label


In [8]:
#------------------------------------------------------------------------------
# 5.2 Create funnel_features, items_session, and orders
#------------------------------------------------------------------------------
print_banner("5.2 Create funnel_features, items_session, and orders")

# Create pre-purchase funnel flags and timing features
funnel_features_sql = f"""
CREATE OR REPLACE TABLE `{FUNNEL_FEATURES_TABLE}` AS
WITH first_purchase AS (
  SELECT user_pseudo_id, ga_session_id, MIN(event_timestamp) AS first_purchase_ts
  FROM `{EVENTS_BASE_TABLE}`
  WHERE ga_session_id IS NOT NULL AND event_name = 'purchase'
  GROUP BY user_pseudo_id, ga_session_id
),
pre_purchase_steps AS (
  SELECT
    e.user_pseudo_id,
    e.ga_session_id,
    e.event_name,
    MIN(e.event_timestamp) AS first_step_ts
  FROM `{EVENTS_BASE_TABLE}` e
  LEFT JOIN first_purchase p
    ON e.user_pseudo_id = p.user_pseudo_id AND e.ga_session_id = p.ga_session_id
  WHERE e.ga_session_id IS NOT NULL
    AND e.event_name IN ('view_item','add_to_cart','begin_checkout','add_shipping_info','add_payment_info')
    AND (p.first_purchase_ts IS NULL OR e.event_timestamp < p.first_purchase_ts)
  GROUP BY e.user_pseudo_id, e.ga_session_id, e.event_name
),
step_pivot AS (
  SELECT
    user_pseudo_id,
    ga_session_id,
    MAX(CASE WHEN event_name = 'view_item' THEN first_step_ts END) AS view_item_ts,
    MAX(CASE WHEN event_name = 'add_to_cart' THEN first_step_ts END) AS add_to_cart_ts,
    MAX(CASE WHEN event_name = 'begin_checkout' THEN first_step_ts END) AS begin_checkout_ts,
    MAX(CASE WHEN event_name = 'add_shipping_info' THEN first_step_ts END) AS add_shipping_info_ts,
    MAX(CASE WHEN event_name = 'add_payment_info' THEN first_step_ts END) AS add_payment_info_ts
  FROM pre_purchase_steps
  GROUP BY user_pseudo_id, ga_session_id
)
SELECT
  s.user_pseudo_id,
  s.ga_session_id,
  IF(p.view_item_ts IS NULL, 0, 1) AS reached_view_item,
  IF(p.add_to_cart_ts IS NULL, 0, 1) AS reached_add_to_cart,
  IF(p.begin_checkout_ts IS NULL, 0, 1) AS reached_begin_checkout,
  IF(p.add_shipping_info_ts IS NULL, 0, 1) AS reached_add_shipping_info,
  IF(p.add_payment_info_ts IS NULL, 0, 1) AS reached_add_payment_info,
  IF(fp.first_purchase_ts IS NULL, 0, 1) AS reached_purchase,
  CASE WHEN p.view_item_ts IS NOT NULL AND p.add_to_cart_ts IS NOT NULL AND p.add_to_cart_ts >= p.view_item_ts THEN SAFE_DIVIDE(p.add_to_cart_ts - p.view_item_ts, 1000000.0) END AS t_view_to_cart_sec,
  CASE WHEN p.add_to_cart_ts IS NOT NULL AND p.begin_checkout_ts IS NOT NULL AND p.begin_checkout_ts >= p.add_to_cart_ts THEN SAFE_DIVIDE(p.begin_checkout_ts - p.add_to_cart_ts, 1000000.0) END AS t_cart_to_checkout_sec,
  CASE WHEN p.begin_checkout_ts IS NOT NULL AND p.add_shipping_info_ts IS NOT NULL AND p.add_shipping_info_ts >= p.begin_checkout_ts THEN SAFE_DIVIDE(p.add_shipping_info_ts - p.begin_checkout_ts, 1000000.0) END AS t_checkout_to_shipping_sec,
  CASE WHEN p.add_shipping_info_ts IS NOT NULL AND p.add_payment_info_ts IS NOT NULL AND p.add_payment_info_ts >= p.add_shipping_info_ts THEN SAFE_DIVIDE(p.add_payment_info_ts - p.add_shipping_info_ts, 1000000.0) END AS t_shipping_to_payment_sec,
  CAST(NULL AS FLOAT64) AS t_payment_to_purchase_sec
FROM `{SESSIONS_TABLE}` s
LEFT JOIN step_pivot p
  ON s.user_pseudo_id = p.user_pseudo_id AND s.ga_session_id = p.ga_session_id
LEFT JOIN first_purchase fp
  ON s.user_pseudo_id = fp.user_pseudo_id AND s.ga_session_id = fp.ga_session_id;
"""

save_sql_and_run("funnel_features_create_table.sql", funnel_features_sql)
print("Created table ->", FUNNEL_FEATURES_TABLE)

# Create item features from pre-purchase item arrays
items_session_sql = f"""
CREATE OR REPLACE TABLE `{ITEMS_SESSION_TABLE}` AS
WITH first_purchase AS (
  SELECT user_pseudo_id, ga_session_id, MIN(event_timestamp) AS first_purchase_ts
  FROM `{EVENTS_BASE_TABLE}`
  WHERE ga_session_id IS NOT NULL AND event_name = 'purchase'
  GROUP BY user_pseudo_id, ga_session_id
),
item_rows AS (
  SELECT
    e.user_pseudo_id,
    e.ga_session_id,
    e.event_timestamp,
    i.item_id,
    i.item_name,
    i.item_brand,
    i.item_category,
    SAFE_CAST(i.price AS FLOAT64) AS item_price,
    COALESCE(i.quantity, 1) AS quantity
  FROM `{EVENTS_BASE_TABLE}` e
  LEFT JOIN first_purchase p
    ON e.user_pseudo_id = p.user_pseudo_id AND e.ga_session_id = p.ga_session_id
  CROSS JOIN UNNEST(e.items) AS i
  WHERE e.ga_session_id IS NOT NULL
    AND ARRAY_LENGTH(e.items) > 0
    AND (p.first_purchase_ts IS NULL OR e.event_timestamp < p.first_purchase_ts)
),
item_agg AS (
  SELECT
    user_pseudo_id,
    ga_session_id,
    COUNT(*) AS n_item_rows,
    COUNT(DISTINCT COALESCE(item_id, item_name, item_category)) AS n_distinct_item_id,
    COUNT(DISTINCT item_category) AS n_distinct_item_category,
    AVG(item_price) AS avg_item_price,
    MAX(item_price) AS max_item_price,
    SUM(COALESCE(quantity, 1)) AS qty_sum,
    ARRAY_AGG(item_brand IGNORE NULLS ORDER BY event_timestamp ASC LIMIT 1)[SAFE_OFFSET(0)] AS example_brand,
    ARRAY_AGG(item_category IGNORE NULLS ORDER BY event_timestamp ASC LIMIT 1)[SAFE_OFFSET(0)] AS example_category
  FROM item_rows
  GROUP BY user_pseudo_id, ga_session_id
)
SELECT
  s.user_pseudo_id,
  s.ga_session_id,
  COALESCE(a.n_item_rows, 0) AS n_item_rows,
  COALESCE(a.n_distinct_item_id, 0) AS n_distinct_item_id,
  COALESCE(a.n_distinct_item_category, 0) AS n_distinct_item_category,
  COALESCE(a.avg_item_price, 0.0) AS avg_item_price,
  COALESCE(a.max_item_price, 0.0) AS max_item_price,
  COALESCE(a.qty_sum, 0) AS qty_sum,
  a.example_brand,
  a.example_category
FROM `{SESSIONS_TABLE}` s
LEFT JOIN item_agg a
  ON s.user_pseudo_id = a.user_pseudo_id AND s.ga_session_id = a.ga_session_id;
"""

save_sql_and_run("items_session_create_table.sql", items_session_sql)
print("Created table ->", ITEMS_SESSION_TABLE)

# Keep orders as purchase level
orders_sql = f"""
CREATE OR REPLACE TABLE `{ORDERS_TABLE}` AS
WITH purchase_events AS (
  SELECT
    user_pseudo_id,
    ga_session_id,
    event_timestamp,
    TIMESTAMP_MICROS(event_timestamp) AS purchase_time,
    DATE(TIMESTAMP_MICROS(event_timestamp)) AS purchase_date,
    DATE_TRUNC(DATE(TIMESTAMP_MICROS(event_timestamp)), WEEK(MONDAY)) AS week_start_date,
    ecommerce.transaction_id AS transaction_id,
    ecommerce.purchase_revenue AS purchase_revenue,
    ecommerce.total_item_quantity AS total_item_quantity,
    ecommerce.tax_value AS tax_value,
    ecommerce.shipping_value AS shipping_value,
    ecommerce.unique_items AS unique_items
  FROM `{EVENTS_BASE_TABLE}`
  WHERE ga_session_id IS NOT NULL AND event_name = 'purchase'
)
SELECT *
FROM purchase_events
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY user_pseudo_id, ga_session_id, COALESCE(transaction_id, CAST(event_timestamp AS STRING))
  ORDER BY event_timestamp) = 1;
"""

save_sql_and_run("orders_create_table.sql", orders_sql)
print("Created table ->", ORDERS_TABLE)



------------------------------------------------------------------------------
5.2 Create funnel_features, items_session, and orders
------------------------------------------------------------------------------
Saved SQL -> /content/drive/MyDrive/Capstone_Project/outputs/sql/funnel_features_create_table.sql
Created table -> ga4-project-490603.ga4_capstone.funnel_features
Saved SQL -> /content/drive/MyDrive/Capstone_Project/outputs/sql/items_session_create_table.sql
Created table -> ga4-project-490603.ga4_capstone.items_session
Saved SQL -> /content/drive/MyDrive/Capstone_Project/outputs/sql/orders_create_table.sql
Created table -> ga4-project-490603.ga4_capstone.orders


In [9]:
#------------------------------------------------------------------------------
# 5.3 Validate that modelling features stop before first purchase
#------------------------------------------------------------------------------
print_banner("5.3 Validate pre-purchase modelling features")

# Compare feature tables with expected pre-purchase counts
pre_purchase_validation_sql = f"""
WITH first_purchase AS (
  SELECT user_pseudo_id, ga_session_id, MIN(event_timestamp) AS first_purchase_ts
  FROM `{EVENTS_BASE_TABLE}`
  WHERE ga_session_id IS NOT NULL AND event_name = 'purchase'
  GROUP BY user_pseudo_id, ga_session_id
),
expected_event_counts AS (
  SELECT
    e.user_pseudo_id,
    e.ga_session_id,
    COUNT(*) AS expected_pre_purchase_events
  FROM `{EVENTS_BASE_TABLE}` e
  LEFT JOIN first_purchase p
    ON e.user_pseudo_id = p.user_pseudo_id AND e.ga_session_id = p.ga_session_id
  WHERE e.ga_session_id IS NOT NULL
    AND (p.first_purchase_ts IS NULL OR e.event_timestamp < p.first_purchase_ts)
  GROUP BY e.user_pseudo_id, e.ga_session_id
),
expected_item_counts AS (
  SELECT
    e.user_pseudo_id,
    e.ga_session_id,
    COUNT(*) AS expected_pre_purchase_item_rows
  FROM `{EVENTS_BASE_TABLE}` e
  LEFT JOIN first_purchase p
    ON e.user_pseudo_id = p.user_pseudo_id AND e.ga_session_id = p.ga_session_id
  CROSS JOIN UNNEST(e.items) AS i
  WHERE e.ga_session_id IS NOT NULL
    AND ARRAY_LENGTH(e.items) > 0
    AND (p.first_purchase_ts IS NULL OR e.event_timestamp < p.first_purchase_ts)
  GROUP BY e.user_pseudo_id, e.ga_session_id
),
session_check AS (
  SELECT
    COUNT(*) AS n_session_rows_checked,
    COUNTIF(s.n_events != COALESCE(e.expected_pre_purchase_events, 0)) AS n_session_event_count_mismatches
  FROM `{SESSIONS_TABLE}` s
  LEFT JOIN expected_event_counts e
    ON s.user_pseudo_id = e.user_pseudo_id AND s.ga_session_id = e.ga_session_id
),
item_check AS (
  SELECT
    COUNT(*) AS n_item_rows_checked,
    COUNTIF(i.n_item_rows != COALESCE(e.expected_pre_purchase_item_rows, 0)) AS n_item_count_mismatches
  FROM `{ITEMS_SESSION_TABLE}` i
  LEFT JOIN expected_item_counts e
    ON i.user_pseudo_id = e.user_pseudo_id AND i.ga_session_id = e.ga_session_id
),
funnel_check AS (
  SELECT COUNT(*) AS n_post_purchase_funnel_steps
  FROM `{FUNNEL_STEPS_TABLE}` f
  JOIN first_purchase p
    ON f.user_pseudo_id = p.user_pseudo_id AND f.ga_session_id = p.ga_session_id
  WHERE f.funnel_step != 'purchase'
    AND f.first_step_ts >= p.first_purchase_ts
)
SELECT
  s.n_session_rows_checked,
  s.n_session_event_count_mismatches,
  i.n_item_rows_checked,
  i.n_item_count_mismatches,
  f.n_post_purchase_funnel_steps
FROM session_check s
CROSS JOIN item_check i
CROSS JOIN funnel_check f;
"""

# Run the pre-purchase validation query
pre_purchase_validation_df = run_bq(pre_purchase_validation_sql)
display(pre_purchase_validation_df)

# Save the validation result
pre_purchase_validation_path = DQ_DIR / "pre_purchase_feature_validation.json"
save_json_overwrite(pre_purchase_validation_path, pre_purchase_validation_df.iloc[0].to_dict())



------------------------------------------------------------------------------
5.3 Validate pre-purchase modelling features
------------------------------------------------------------------------------


,n_session_rows_checked,n_session_event_count_mismatches,n_item_rows_checked,n_item_count_mismatches,n_post_purchase_funnel_steps
0,360129,0,360129,0,0


# 6.0 Create weekly Power BI fact tables

The weekly fact tables provide the first dashboard ready outputs from the gold table layer. They aggregate session volume, purchases, engagement, and funnel step reach at a weekly segment grain so Notebook 02 and Power BI can analyse trends without importing large event-level tables.

**Process**:
1. Aggregate sessions by week, device category, country, source, medium, and channel segment.
2. Calculate weekly session counts, purchase counts, purchase rate, average session duration, and average engagement time.
3. Aggregate funnel step reach by the same weekly segment grain so step level conversion diagnostics can be built consistently later.
4. Export the two weekly fact tables as CSV files to the Power BI latest folder.
5. Display previews of both fact tables to confirm the weekly segment grain and column structure.

In [10]:
#------------------------------------------------------------------------------
# 6.1 Create weekly reporting tables
#------------------------------------------------------------------------------
print_banner("6.1 Create weekly reporting tables")

# Aggregate weekly session and purchase metrics
fact_sessions_weekly_sql = f"""
CREATE OR REPLACE TABLE `{FACT_SESSIONS_WEEKLY_TABLE}` AS
SELECT
  s.week_start_date,
  s.device_category,
  s.geo_country,
  s.channel_source,
  s.channel_medium,
  s.channel_segment,
  COUNT(*) AS n_sessions,
  SUM(COALESCE(l.y_purchase, 0)) AS n_purchases,
  SAFE_DIVIDE(SUM(COALESCE(l.y_purchase, 0)), COUNT(*)) AS purchase_rate,
  AVG(s.session_duration_seconds) AS avg_session_duration_sec,
  AVG(s.engagement_time_msec_sum) AS avg_engagement_time_msec
FROM `{SESSIONS_TABLE}` s
LEFT JOIN `{SESSION_LABEL_TABLE}` l
  ON s.user_pseudo_id = l.user_pseudo_id AND s.ga_session_id = l.ga_session_id
GROUP BY s.week_start_date, s.device_category, s.geo_country, s.channel_source, s.channel_medium, s.channel_segment;
"""

save_sql_and_run("fact_sessions_weekly_create_table.sql", fact_sessions_weekly_sql)
print("Created table ->", FACT_SESSIONS_WEEKLY_TABLE)

# Aggregate weekly funnel-step reach
fact_funnel_weekly_sql = f"""
CREATE OR REPLACE TABLE `{FACT_FUNNEL_WEEKLY_TABLE}` AS
SELECT
  s.week_start_date,
  s.device_category,
  s.geo_country,
  s.channel_source,
  s.channel_medium,
  s.channel_segment,
  f.funnel_step,
  COUNT(*) AS n_sessions_reaching_step
FROM `{SESSIONS_TABLE}` s
JOIN `{FUNNEL_STEPS_TABLE}` f
  ON s.user_pseudo_id = f.user_pseudo_id AND s.ga_session_id = f.ga_session_id
GROUP BY s.week_start_date, s.device_category, s.geo_country, s.channel_source, s.channel_medium, s.channel_segment, f.funnel_step;
"""

save_sql_and_run("fact_funnel_weekly_create_table.sql", fact_funnel_weekly_sql)
print("Created table ->", FACT_FUNNEL_WEEKLY_TABLE)



------------------------------------------------------------------------------
6.1 Create weekly reporting tables
------------------------------------------------------------------------------
Saved SQL -> /content/drive/MyDrive/Capstone_Project/outputs/sql/fact_sessions_weekly_create_table.sql
Created table -> ga4-project-490603.ga4_capstone.fact_sessions_weekly
Saved SQL -> /content/drive/MyDrive/Capstone_Project/outputs/sql/fact_funnel_weekly_create_table.sql
Created table -> ga4-project-490603.ga4_capstone.fact_funnel_weekly


In [11]:
#------------------------------------------------------------------------------
# 6.2 Export weekly Power BI CSV tables
#------------------------------------------------------------------------------
print_banner("6.2 Export weekly Power BI CSV tables")

# Export weekly session facts for Power BI
fact_sessions_weekly_df = export_query(
    f"""SELECT * FROM `{FACT_SESSIONS_WEEKLY_TABLE}`
    ORDER BY week_start_date, device_category, geo_country, channel_segment""",
    csv_path=PBI_LATEST_DIR / "fact_sessions_weekly.csv",
    date_cols=["week_start_date"])

# Export weekly funnel facts for Power BI
fact_funnel_weekly_df = export_query(
    f"""SELECT * FROM `{FACT_FUNNEL_WEEKLY_TABLE}`
    ORDER BY week_start_date, device_category, geo_country, channel_segment, funnel_step""",
    csv_path=PBI_LATEST_DIR / "fact_funnel_weekly.csv",
    date_cols=["week_start_date"])

display(fact_sessions_weekly_df.head(10))
display(fact_funnel_weekly_df.head(10))



------------------------------------------------------------------------------
6.2 Export weekly Power BI CSV tables
------------------------------------------------------------------------------


,week_start_date,device_category,geo_country,channel_source,channel_medium,channel_segment,n_sessions,n_purchases,purchase_rate,avg_session_duration_sec,avg_engagement_time_msec
0,2020-10-26,desktop,(not set),(direct),(none),(direct) / (none),1,0,0.0000,5.7318,"5,313.0000"
1,2020-10-26,desktop,(not set),<Other>,<Other>,<Other> / <Other>,3,0,0.0000,55.8405,"49,599.3333"
2,2020-10-26,desktop,(not set),<Other>,organic,<Other> / organic,1,0,0.0000,16.5658,"16,439.0000"
3,2020-10-26,desktop,(not set),<Other>,referral,<Other> / referral,1,0,0.0000,853.4483,"40,209.0000"
4,2020-10-26,desktop,(not set),google,organic,google / organic,3,0,0.0000,156.9571,"7,882.6667"
5,2020-10-26,desktop,(not set),shop.googlemerchandisestore.com,referral,shop.googlemerchandisestore.com / referral,2,0,0.0000,17.6292,"10,217.0000"
6,2020-10-26,desktop,Albania,<Other>,<Other>,<Other> / <Other>,1,0,0.0000,0.0000,0.0000
7,2020-10-26,desktop,Argentina,google,organic,google / organic,1,0,0.0000,322.7984,"279,718.0000"
8,2020-10-26,desktop,Argentina,shop.googlemerchandisestore.com,referral,shop.googlemerchandisestore.com / referral,1,0,0.0000,21.9749,"19,114.0000"
9,2020-10-26,desktop,Australia,(direct),(none),(direct) / (none),3,0,0.0000,17.3327,"15,629.0000"


,week_start_date,device_category,geo_country,channel_source,channel_medium,channel_segment,funnel_step,n_sessions_reaching_step
0,2020-10-26,desktop,(not set),<Other>,<Other>,<Other> / <Other>,view_item,1
1,2020-10-26,desktop,(not set),<Other>,referral,<Other> / referral,add_shipping_info,1
2,2020-10-26,desktop,(not set),<Other>,referral,<Other> / referral,begin_checkout,1
3,2020-10-26,desktop,(not set),<Other>,referral,<Other> / referral,view_item,1
4,2020-10-26,desktop,(not set),google,organic,google / organic,view_item,1
5,2020-10-26,desktop,Argentina,shop.googlemerchandisestore.com,referral,shop.googlemerchandisestore.com / referral,view_item,1
6,2020-10-26,desktop,Austria,google,organic,google / organic,view_item,1
7,2020-10-26,desktop,Bangladesh,(direct),(none),(direct) / (none),view_item,1
8,2020-10-26,desktop,Belgium,<Other>,referral,<Other> / referral,view_item,1
9,2020-10-26,desktop,Belgium,shop.googlemerchandisestore.com,referral,shop.googlemerchandisestore.com / referral,view_item,1


# 7.0 Export modelling inputs and data quality checks

The export step saves the structured files needed by the later notebooks and records checks for the final build. These checks confirm table counts, grain alignment, funnel consistency, pre-purchase validation, and output locations.

**Process**:
1. Export `sessions.parquet`, `session_label.parquet`, `funnel_features.parquet`, and `items_session.parquet` to the processed data folder.
2. Run table count and grain checks to confirm the key session-level tables align with the `sessions` table.
3. Run funnel reach and uniqueness checks to confirm the funnel outputs and order table behave as expected.
4. Save a table dictionary that documents each final BigQuery table, its grain, purpose, and row count.
5. Save `gold_tables_sanity.json` with the main checks and the pre-purchase validation result for reproducibility.


In [12]:
#------------------------------------------------------------------------------
# 7.1 Export required processed parquet tables
#------------------------------------------------------------------------------
print_banner("7.1 Export required processed parquet tables")

# List the processed parquet outputs needed downstream
processed_exports = [
    ("sessions.parquet", f"SELECT * FROM `{SESSIONS_TABLE}` ORDER BY session_start_time", ["session_date","week_start_date"], ["session_start_time","session_end_time"], ["user_pseudo_id","user_session_key"]),
    ("session_label.parquet", f"SELECT * FROM `{SESSION_LABEL_TABLE}`", [], [], ["user_pseudo_id"]),
    ("funnel_features.parquet", f"SELECT * FROM `{FUNNEL_FEATURES_TABLE}`", [], [], ["user_pseudo_id"]),
    ("items_session.parquet", f"SELECT * FROM `{ITEMS_SESSION_TABLE}`", [], [], ["user_pseudo_id"])]

# Export each processed table to Drive
for file_name, sql, date_cols, datetime_cols, string_cols in processed_exports:
    df = export_query(sql, parquet_path=PROCESSED_DIR / file_name, date_cols=date_cols, datetime_cols=datetime_cols, string_cols=string_cols)
    print(f"{file_name}: {df.shape[0]:,} rows, {df.shape[1]:,} columns")

print("Processed parquet exports created in ->", PROCESSED_DIR)



------------------------------------------------------------------------------
7.1 Export required processed parquet tables
------------------------------------------------------------------------------
sessions.parquet: 360,129 rows, 26 columns
session_label.parquet: 360,129 rows, 3 columns
funnel_features.parquet: 360,129 rows, 13 columns
items_session.parquet: 360,129 rows, 10 columns
Processed parquet exports created in -> /content/drive/MyDrive/Capstone_Project/data/processed


In [13]:
#------------------------------------------------------------------------------
# 7.2 Create data quality checks
#------------------------------------------------------------------------------
print_banner("7.2 Create data quality checks")

# Count rows in each final BigQuery table
table_counts_sql = f"""
SELECT 'events_base' AS table_name, COUNT(*) AS n_rows FROM `{EVENTS_BASE_TABLE}`
UNION ALL SELECT 'sessions', COUNT(*) FROM `{SESSIONS_TABLE}`
UNION ALL SELECT 'funnel_steps', COUNT(*) FROM `{FUNNEL_STEPS_TABLE}`
UNION ALL SELECT 'session_label', COUNT(*) FROM `{SESSION_LABEL_TABLE}`
UNION ALL SELECT 'funnel_features', COUNT(*) FROM `{FUNNEL_FEATURES_TABLE}`
UNION ALL SELECT 'items_session', COUNT(*) FROM `{ITEMS_SESSION_TABLE}`
UNION ALL SELECT 'orders', COUNT(*) FROM `{ORDERS_TABLE}`
UNION ALL SELECT 'fact_sessions_weekly', COUNT(*) FROM `{FACT_SESSIONS_WEEKLY_TABLE}`
UNION ALL SELECT 'fact_funnel_weekly', COUNT(*) FROM `{FACT_FUNNEL_WEEKLY_TABLE}`;
"""

# Check session-level table row alignment
grain_checks_sql = f"""
SELECT
  (SELECT COUNT(*) FROM `{SESSIONS_TABLE}`) AS n_sessions,
  (SELECT COUNT(*) FROM `{SESSION_LABEL_TABLE}`) AS n_session_label_rows,
  (SELECT COUNT(*) FROM `{FUNNEL_FEATURES_TABLE}`) AS n_funnel_features_rows,
  (SELECT COUNT(*) FROM `{ITEMS_SESSION_TABLE}`) AS n_items_session_rows,
  (SELECT COUNT(*) FROM `{ORDERS_TABLE}`) AS n_orders;
"""

# Summarise funnel step reach counts
funnel_reach_sql = f"""
SELECT
  SUM(reached_view_item) AS sessions_reached_view_item,
  SUM(reached_add_to_cart) AS sessions_reached_add_to_cart,
  SUM(reached_begin_checkout) AS sessions_reached_begin_checkout,
  SUM(reached_add_shipping_info) AS sessions_reached_add_shipping_info,
  SUM(reached_add_payment_info) AS sessions_reached_add_payment_info,
  SUM(reached_purchase) AS sessions_reached_purchase
FROM `{FUNNEL_FEATURES_TABLE}`;
"""

# Check duplicate session and order keys
integrity_checks_sql = f"""
WITH session_dupes AS (
  SELECT user_pseudo_id, ga_session_id, COUNT(*) AS n_rows
  FROM `{SESSIONS_TABLE}`
  GROUP BY user_pseudo_id, ga_session_id
  HAVING COUNT(*) > 1
),
order_dupes AS (
  SELECT user_pseudo_id, ga_session_id, COALESCE(transaction_id, CAST(event_timestamp AS STRING)) AS final_purchase_key, COUNT(*) AS n_rows
  FROM `{ORDERS_TABLE}`
  GROUP BY user_pseudo_id, ga_session_id, COALESCE(transaction_id, CAST(event_timestamp AS STRING))
  HAVING COUNT(*) > 1
)
SELECT
  (SELECT COUNT(*) FROM session_dupes) AS n_duplicate_session_key_groups,
  (SELECT COALESCE(SUM(n_rows - 1), 0) FROM session_dupes) AS n_extra_session_rows,
  (SELECT COUNT(*) FROM order_dupes) AS n_duplicate_order_key_groups,
  (SELECT COALESCE(SUM(n_rows - 1), 0) FROM order_dupes) AS n_extra_order_rows;
"""

# Run the final quality check queries
table_counts_df = run_bq(table_counts_sql)
grain_checks_df = run_bq(grain_checks_sql)
funnel_reach_df = run_bq(funnel_reach_sql)
integrity_checks_df = run_bq(integrity_checks_sql)

display(table_counts_df)
display(grain_checks_df)
display(funnel_reach_df)
display(integrity_checks_df)



------------------------------------------------------------------------------
7.2 Create data quality checks
------------------------------------------------------------------------------


,table_name,n_rows
0,sessions,360129
1,events_base,4295584
2,fact_sessions_weekly,22457
3,funnel_steps,126026
4,fact_funnel_weekly,27607
5,items_session,360129
6,funnel_features,360129
7,orders,5281
8,session_label,360129


,n_sessions,n_session_label_rows,n_funnel_features_rows,n_items_session_rows,n_orders
0,360129,360129,360129,360129,5281


,sessions_reached_view_item,sessions_reached_add_to_cart,sessions_reached_begin_checkout,sessions_reached_add_shipping_info,sessions_reached_add_payment_info,sessions_reached_purchase
0,76995,15157,11106,11105,6815,4848


,n_duplicate_session_key_groups,n_extra_session_rows,n_duplicate_order_key_groups,n_extra_order_rows
0,0,0,0,0


In [14]:
#------------------------------------------------------------------------------
# 7.3 Save table dictionary and gold-table sanity report
#------------------------------------------------------------------------------
print_banner("7.3 Save table dictionary and gold-table sanity report")

# Map table names to row counts
table_count_map = dict(zip(table_counts_df["table_name"], table_counts_df["n_rows"]))

# Define the table dictionary rows
table_rows = [
    ("events_base", "event", "Flattened event-level staging table used to build later tables"),
    ("sessions", "user_pseudo_id + ga_session_id", "Core pre-purchase session behaviour and segment table"),
    ("funnel_steps", "session + funnel_step", "First reach timestamp for each funnel milestone, excluding post-purchase repeats"),
    ("session_label", "session", "Binary purchase label used by modelling"),
    ("funnel_features", "session", "Pre-purchase funnel reach flags and step timing features"),
    ("items_session", "session", "Pre-purchase session-level item and product-interest summary"),
    ("orders", "purchase record", "Deduplicated purchase records for audit and reporting"),
    ("fact_sessions_weekly", "week + segment", "Weekly session, purchase, and engagement fact table"),
    ("fact_funnel_weekly", "week + segment + funnel_step", "Weekly funnel-step reach fact table")]

table_dictionary_df = pd.DataFrame([{
    "table_name": table_name,
    "grain": grain,
    "purpose": purpose,
    "n_rows": int(table_count_map.get(table_name, 0))}
    for table_name, grain, purpose in table_rows])

# Save the table dictionary
table_dictionary_path = DQ_DIR / "table_dictionary.csv"
table_dictionary_df.to_csv(table_dictionary_path, index=False)

# Collect check rows for the final sanity report
grain_row = grain_checks_df.iloc[0].to_dict()
funnel_row = funnel_reach_df.iloc[0].to_dict()
integrity_row = integrity_checks_df.iloc[0].to_dict()
pre_purchase_row = pre_purchase_validation_df.iloc[0].to_dict()

# Build the final sanity payload
gold_tables_sanity = {
    "date_window": {"START_DATE": START_DATE, "END_DATE": END_DATE},
    "table_counts": table_counts_df.to_dict(orient="records"),
    "grain_checks": grain_row,
    "funnel_reach_counts": funnel_row,
    "integrity_checks": integrity_row,
    "processed_outputs_kept": [x[0] for x in processed_exports],
    "powerbi_outputs_kept": ["fact_sessions_weekly.csv", "fact_funnel_weekly.csv"],
    "session_label_matches_sessions": int(grain_row["n_sessions"]) == int(grain_row["n_session_label_rows"]),
    "funnel_features_matches_sessions": int(grain_row["n_sessions"]) == int(grain_row["n_funnel_features_rows"]),
    "items_session_matches_sessions": int(grain_row["n_sessions"]) == int(grain_row["n_items_session_rows"]),
    "sessions_unique_on_user_session_key": int(integrity_row["n_duplicate_session_key_groups"]) == 0 and int(integrity_row["n_extra_session_rows"]) == 0,
    "orders_unique_on_final_purchase_key": int(integrity_row["n_duplicate_order_key_groups"]) == 0 and int(integrity_row["n_extra_order_rows"]) == 0,
    "pre_purchase_feature_validation": pre_purchase_row,
    "pre_purchase_feature_validation_passed": (
        int(pre_purchase_row.get("n_session_event_count_mismatches", 0)) == 0 and
        int(pre_purchase_row.get("n_item_count_mismatches", 0)) == 0 and
        int(pre_purchase_row.get("n_post_purchase_funnel_steps", 0)) == 0),
    "funnel_monotonic_until_payment": (
        float(funnel_row["sessions_reached_view_item"]) >= float(funnel_row["sessions_reached_add_to_cart"]) >=
        float(funnel_row["sessions_reached_begin_checkout"]) >= float(funnel_row["sessions_reached_add_shipping_info"]) >=
        float(funnel_row["sessions_reached_add_payment_info"])),
    "purchase_sessions_can_skip_prior_steps": float(funnel_row["sessions_reached_purchase"]) > float(funnel_row["sessions_reached_add_payment_info"]),
    "table_dictionary_file": str(table_dictionary_path)}

# Save the sanity report to Drive
save_json_overwrite(DQ_DIR / "gold_tables_sanity.json", gold_tables_sanity)

display(table_dictionary_df)
print("Saved table dictionary ->", table_dictionary_path)
print("Saved sanity report    ->", DQ_DIR / "gold_tables_sanity.json")



------------------------------------------------------------------------------
7.3 Save table dictionary and gold-table sanity report
------------------------------------------------------------------------------


,table_name,grain,purpose,n_rows
0,events_base,event,Flattened event-level staging table used to build later tables,4295584
1,sessions,user_pseudo_id + ga_session_id,Core pre-purchase session behaviour and segment table,360129
2,funnel_steps,session + funnel_step,"First reach timestamp for each funnel milestone, excluding post-purchase repeats",126026
3,session_label,session,Binary purchase label used by modelling,360129
4,funnel_features,session,Pre-purchase funnel reach flags and step timing features,360129
5,items_session,session,Pre-purchase session-level item and product-interest summary,360129
6,orders,purchase record,Deduplicated purchase records for audit and reporting,5281
7,fact_sessions_weekly,week + segment,"Weekly session, purchase, and engagement fact table",22457
8,fact_funnel_weekly,week + segment + funnel_step,Weekly funnel-step reach fact table,27607


Saved table dictionary -> /content/drive/MyDrive/Capstone_Project/outputs/data_quality/table_dictionary.csv
Saved sanity report    -> /content/drive/MyDrive/Capstone_Project/outputs/data_quality/gold_tables_sanity.json
